# Main Prototype

## Install dependencies

### Check Python and Cuda versions

In [1]:
import sys
print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [2]:
# check NVCC version
!nvcc -V

# check GCC version
!gcc --version

# check python in conda environment
!which python

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

/usr/local/bin/python


### Google Colab

In [ ]:
!git clone https://github.com/Keeron1/skeleton-based-hoi-recognition.git
%cd /content/skeleton-based-hoi-recognition

In [ ]:
%pip install -r requirements.txt

#### Install MMPose

In [ ]:
%pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 \ --index-url https://download.pytorch.org/whl/cu118

In [ ]:
%pip install numpy==1.26.4

In [ ]:
!pip3 install openmim
!mim install mmengine
!mim install "mmcv==2.2.0"
!mim install mmpose

In [ ]:
import torch
import mmcv
import mmengine
import mmpose

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MMCV:", mmcv.__version__)
print("MMEngine:", mmengine.__version__)
print("MMPose imported successfully")

### Local
Run the powershell script to setup the local environment

## Run Imports

In [5]:
import sys
from pathlib import Path
import os
import cv2
import glob

In [6]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from utils.config.config import Config
from src.utils.draw_boxes import DrawBoxes
from src.detector.yolo_detector import YOLODetector
from src.tracker.deepsort_tracker import DeepSortTracker
from src.pose_estimator.hrnet_pose import HrNetPose

## Config


### Setup Env variables for Google Colab

In [ ]:
# Set Colab env vars
os.environ["ENV"] = "colab"
os.environ["DATA_ROOT"] = "/content/drive/MyDrive/HOI-Dataset"
from google.colab.patches import cv2_imshow # cv2.imshow("title", frame) doesn't work in Colab so this is the fix cv2_imshow(frame)

### Initalize config

In [ ]:
config_loader = Config()
cfg = config_loader.load_config()

## Dataset

## YOLO

In [ ]:
# Load model
detector = YOLODetector(
    model_type_path=cfg.yolo.model_path,
    project_root=cfg.project_root,
)

### Train

In [ ]:
results = YOLODetector.train(cfg.paths.dataset)

### Test

In [ ]:
frame = cv2.imread("test.jpg")
results = detector.predict(frame)

print(results)

In [ ]:
# Process results list
for result in results[:10]:
    boxes = result.boxes  # Boxes object for bounding box outputs
    # print(boxes.data.tolist()) # x1, y1, x2, y2, conf score, class id
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen
    # result.save(filename="result.jpg")  # save to disk

## DeepSORT

### Init

In [ ]:
tracker = DeepSortTracker(cfg.deepsort.max_age,
                          cfg.deepsort.n_init,
                          cfg.deepsort.nn_budget,
                          cfg.deepsort.max_cosine_distance,
                          cfg.deepsort.max_iou_distance,
                          cfg.deepsort.nms_max_overlap
                        )

## Pose Estimation

### Init

In [ ]:
pose_estimator = HrNetPose(cfg.hrnet.model_cfg, cfg.hrnet.model_ckpt, cfg.device)

## Main

In [ ]:
def run_main():
    # frame....
    
    # Run DeepSORT
    deepsort_results = tracker.run_deepsort([])

    tracked_bboxes = deepsort_results["bboxes"]
    tracked_ids = deepsort_results["track_ids"]
    # tracked_classes = deepsort_results["class_names"]

    # Skip current frame since there aren't any detected people
    if not tracked_bboxes:
        continue

    # Run HrNet
    pose_results = pose_estimator.infer(frame, tracked_bboxes)

    # Associate each person with their id, bbox, and keypoints
    people = []
    for i in range(len(tracked_ids)):
        people.append({
            "track_id": tracked_ids[i],
            "bbox": tracked_bboxes[i],
            "keypoints": pose_results[i]["keypoints"]
        })